This experiment was conducted based on PubMedQA: A Dataset for Biomedical Research Question Answering. Dataset link: https://github.com/pubmedqa/pubmedqa

# Import Libraries

In [ ]:
import json
import requests
import torch
import numpy as np
from scipy.special import softmax
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch import nn
from openai import OpenAI

import torch.nn as nn
import torch.optim as optim
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd

# HuggingFace Credentials

In [ ]:
! huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineGrained).
The token `pubmedqa` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `pubmedqa`

# Data Loader Configuration

# OpenAI API Client (for GPT models)

In [ ]:
client = OpenAI(api_key="REDACTED_USE_ENV_OPENAI_API_KEY")  # Replace with actual API key

# GPT-2 Model on PubMedQA

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import requests
import json
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.special import softmax

# ✅ Load dataset from GitHub (PubMedQA)
dataset_url = "https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json"
response = requests.get(dataset_url)
data = response.json()

# ✅ Extract questions
questions = [item["QUESTION"] for item in data.values()]
num_samples = min(10, len(questions))  # Limit to 10 samples for evaluation

# ✅ Load GPT-2 Model & Tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # Ensure padding token exists
model = AutoModelForCausalLM.from_pretrained("gpt2")

# ✅ Function: Generate Response (No Uncertainty)
def generate_response_no_uncertainty(input_text):
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    output_sequences = model.generate(inputs["input_ids"], max_new_tokens=50, pad_token_id=tokenizer.eos_token_id)
    response = tokenizer.decode(output_sequences[0], skip_special_tokens=True)
    return response

# ✅ BayesianGPT Class (MC Dropout)
class BayesianGPT(nn.Module):
    def __init__(self, base_model):
        super(BayesianGPT, self).__init__()
        self.base_model = base_model
        self.dropout = nn.Dropout(p=0.3)  # MC Dropout for Bayesian Estimation

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids, attention_mask=attention_mask)
        logits = self.dropout(outputs.logits)  # Apply dropout
        return logits

# ✅ Apply Bayesian Model
bayesian_model = BayesianGPT(model)
bayesian_model.train()

# ✅ Function: Estimate Uncertainty (MC Dropout)
def estimate_uncertainty_gpt(input_text, n_samples=20):
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    input_ids, attention_mask = inputs["input_ids"], inputs["attention_mask"]
    predictions = []

    for _ in range(n_samples):
        with torch.no_grad():
            logits = bayesian_model(input_ids, attention_mask)
            predictions.append(logits.numpy())

    predictions = np.array(predictions)
    mean_prediction = predictions.mean(axis=0)
    uncertainty = np.abs(predictions.std(axis=0) / (mean_prediction + 1e-6))  # Ensure non-negative uncertainty
    return mean_prediction, uncertainty

# ✅ Clinical Uncertainty Score (CUS)
def compute_cus(uncertainties):
    return np.mean(uncertainties)

# ✅ Improved Zero-Shot Trustworthiness Index (ZTI)
def compute_zti(cus_scores, epsilon=1e-6):
    if len(cus_scores) < 2:
        return 1.0  # Default to fully trustworthy if only one sample is available

    mean_cus = np.mean(cus_scores)
    std_cus = np.std(cus_scores)
    max_cus = max(np.max(cus_scores), epsilon)  # Ensure non-zero max for stability

    zti = 1 - (np.log1p(abs(mean_cus - std_cus)) / np.log1p(max_cus))  # Ensure non-negative log values
    return max(0, min(zti, 1))  # Ensure ZTI stays within [0,1] range

# ✅ Compute CUS and ZTI across multiple samples
cus_scores_no_uncertainty = []
cus_scores_with_uncertainty = []

for i in range(num_samples):
    # 🔹 Without Uncertainty
    response_no_uncertainty = generate_response_no_uncertainty(questions[i])
    print(f"Sample {i+1} - GPT Response (No Uncertainty): {response_no_uncertainty}")
    pred_no_uncertainty = model(tokenizer(questions[i], return_tensors="pt")["input_ids"])  # Raw prediction
    cus_score_no_uncertainty = compute_cus(pred_no_uncertainty.logits.detach().numpy())
    cus_scores_no_uncertainty.append(cus_score_no_uncertainty)
    print(f"Sample {i+1} - CUS (No Uncertainty): {cus_score_no_uncertainty}")

    # 🔹 With Uncertainty (Monte Carlo Dropout)
    pred, uncertainty = estimate_uncertainty_gpt(questions[i])
    cus_score = compute_cus(uncertainty)
    cus_scores_with_uncertainty.append(cus_score)
    print(f"Sample {i+1} - CUS (With Uncertainty): {cus_score}")

# ✅ Compute Overall ZTI
zti_score_no_uncertainty = compute_zti(cus_scores_no_uncertainty)
zti_score_with_uncertainty = compute_zti(cus_scores_with_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) No Uncertainty:", zti_score_no_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) With Uncertainty:", zti_score_with_uncertainty)

# ✅ Generate Final Comparison Report
comparison_results = {
    "CUS_No_Uncertainty": cus_scores_no_uncertainty,
    "CUS_With_Uncertainty": cus_scores_with_uncertainty,
    "ZTI_No_Uncertainty": zti_score_no_uncertainty,
    "ZTI_With_Uncertainty": zti_score_with_uncertainty,
}
print("Comparison Results:", comparison_results)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Sample 1 - GPT Response (No Uncertainty): Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
The role of the mitochondria in the cell cycle of the cell isomatoid cycle is not clear. The role of the mitochondria in the cell cycle is not clear. The role of the mitochondria in the cell cycle is not clear
Sample 1 - CUS (No Uncertainty): -92.45392608642578
Sample 1 - CUS (With Uncertainty): 0.679412305355072
Sample 2 - GPT Response (No Uncertainty): Landolt C and snellen e acuity: differences in strabismus amblyopia?

A. The first two words are the same as the first two words.

B. The second two words are the same as the first two words.

C. The first two words are the same as the first two words
Sample 2 - CUS (No Uncertainty): -89.2373275756836
Sample 2 - CUS (With Uncertainty): 0.6888371109962463
Sample 3 - GPT Response (No Uncertainty): Syncope during bathing in infants, a pediatric form of water-induced urticaria?

The answer is yes.

The first 

# GPT-3 on PubMedQA

In [ ]:
import openai
import numpy as np
import json
import time
import requests
import torch
import torch.nn as nn
from openai import OpenAI


# ✅ Load dataset from GitHub (PubMedQA)
dataset_url = "https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json"
response = requests.get(dataset_url)
data = response.json()

# ✅ Extract questions
questions = [item["QUESTION"] for item in data.values()]
num_samples = min(10, len(questions))  # Limit to 10 samples for evaluation

# ✅ Define BayesianGPT3_5 Model for Uncertainty Estimation
class BayesianGPT3_5(nn.Module):
    def __init__(self):
        super(BayesianGPT3_5, self).__init__()
        self.dropout = nn.Dropout(p=0.3)  # Apply MC Dropout (simulated via multiple API calls)

    def forward(self, input_text, n_samples=10):
        """Simulate Bayesian inference with multiple API calls (Monte Carlo Estimation)."""
        predictions = []

        for _ in range(n_samples):
            response = client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[{"role": "user", "content": input_text}],
                max_tokens=50,
                temperature=np.random.uniform(0.7, 1.2)  # Randomized temp for diversity
            )
            predictions.append(response.choices[0].message.content)
            time.sleep(1)  # Avoid rate limit

        # Convert predictions to numerical values
        prediction_lengths = [len(pred.split()) for pred in predictions]
        mean_prediction = np.mean(prediction_lengths)
        uncertainty = np.abs(np.std(prediction_lengths) / (mean_prediction + 1e-6))  # Normalize uncertainty

        return mean_prediction, uncertainty

# ✅ Instantiate Bayesian GPT-3.5 Model
bayesian_model = BayesianGPT3_5()

# ✅ Function: Generate Response Without Uncertainty
def generate_response_no_uncertainty(input_text):
    """Generate response using GPT-3.5 without uncertainty modeling."""
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": input_text}],
        max_tokens=50,
        temperature=1.0
    )
    return response.choices[0].message.content

# ✅ Clinical Uncertainty Score (CUS)
def compute_cus(uncertainties):
    """Compute the Clinical Uncertainty Score (CUS)."""
    return np.mean(uncertainties)

# ✅ Improved Zero-Shot Trustworthiness Index (ZTI)
def compute_zti(cus_scores, epsilon=1e-6):
    """Compute Zero-Shot Trustworthiness Index (ZTI)."""
    if len(cus_scores) < 2:
        return 1.0  # Default to fully trustworthy if only one sample is available

    mean_cus = np.mean(cus_scores)
    std_cus = np.std(cus_scores)
    max_cus = max(np.max(cus_scores), epsilon)  # Ensure non-zero max for stability

    zti = 1 - (np.log1p(abs(mean_cus - std_cus)) / np.log1p(max_cus))  # Logarithmic scaling
    return max(0, min(zti, 1))  # Ensure ZTI stays within [0,1] range

# ✅ Compute CUS and ZTI across multiple samples
cus_scores_no_uncertainty = []
cus_scores_with_uncertainty = []

for i in range(num_samples):
    # 🔹 Without Uncertainty
    response_no_uncertainty = generate_response_no_uncertainty(questions[i])
    print(f"Sample {i+1} - GPT-3.5 Response (No Uncertainty): {response_no_uncertainty}")

    cus_score_no_uncertainty = len(response_no_uncertainty.split())  # Length-based uncertainty proxy
    cus_scores_no_uncertainty.append(cus_score_no_uncertainty)
    print(f"Sample {i+1} - CUS (No Uncertainty): {cus_score_no_uncertainty}")

    # 🔹 With Bayesian Uncertainty Modeling
    mean_pred, uncertainty = bayesian_model(questions[i], n_samples=10)
    cus_score = compute_cus([uncertainty])
    cus_scores_with_uncertainty.append(cus_score)
    print(f"Sample {i+1} - CUS (With Uncertainty): {cus_score}")

# ✅ Compute Overall ZTI
zti_score_no_uncertainty = compute_zti(cus_scores_no_uncertainty)
zti_score_with_uncertainty = compute_zti(cus_scores_with_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) No Uncertainty:", zti_score_no_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) With Uncertainty:", zti_score_with_uncertainty)

# ✅ Generate Final Comparison Report
comparison_results = {
    "CUS_No_Uncertainty": cus_scores_no_uncertainty,
    "CUS_With_Uncertainty": cus_scores_with_uncertainty,
    "ZTI_No_Uncertainty": zti_score_no_uncertainty,
    "ZTI_With_Uncertainty": zti_score_with_uncertainty,
}
print("Comparison Results:", comparison_results)


Sample 1 - GPT-3.5 Response (No Uncertainty): Yes, mitochondria do play a key role in the remodelling of lace plant leaves during programmed cell death. Programmed cell death is a highly regulated process that involves the activation of various cellular pathways, including mitochondrial pathways. Mitochondria are responsible
Sample 1 - CUS (No Uncertainty): 40
Sample 1 - CUS (With Uncertainty): 0.046820905323459526
Sample 2 - GPT-3.5 Response (No Uncertainty): In strabismic amblyopia, there may be differences in Landolt C and Snellen e acuity measurements. Landolt C acuity typically tests visual acuity by presenting a target ring with a gap in a specific orientation, which the patient
Sample 2 - CUS (No Uncertainty): 37
Sample 2 - CUS (With Uncertainty): 0.07988836083220527
Sample 3 - GPT-3.5 Response (No Uncertainty): Syncope during bathing in infants is not typically considered a form of water-induced urticaria. Syncope, or fainting, during bathing can occur for a variety of reasons 

# GPT-4 on PubMedQA

In [ ]:
import openai
import numpy as np
import json
import time
import requests
import torch
import torch.nn as nn
from openai import OpenAI


# ✅ Load dataset from GitHub (PubMedQA)
dataset_url = "https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json"
response = requests.get(dataset_url)
data = response.json()

# ✅ Extract questions
questions = [item["QUESTION"] for item in data.values()]
num_samples = min(10, len(questions))  # Limit to 10 samples for evaluation

# ✅ Define BayesianGPT4 Model for Uncertainty Estimation
class BayesianGPT4(nn.Module):
    def __init__(self):
        super(BayesianGPT4, self).__init__()
        self.dropout = nn.Dropout(p=0.3)  # Apply MC Dropout (simulated via multiple API calls)

    def forward(self, input_text, n_samples=10):
        """Simulate Bayesian inference with multiple API calls (Monte Carlo Estimation)."""
        predictions = []

        for _ in range(n_samples):
            response = client.chat.completions.create(
                model="gpt-4",
                messages=[{"role": "user", "content": input_text}],
                max_tokens=50,
                temperature=np.random.uniform(0.7, 1.2)  # Randomized temp for diversity
            )
            predictions.append(response.choices[0].message.content)
            time.sleep(1)  # Avoid rate limit

        # Convert predictions to numerical values
        prediction_lengths = [len(pred.split()) for pred in predictions]
        mean_prediction = np.mean(prediction_lengths)
        uncertainty = np.abs(np.std(prediction_lengths) / (mean_prediction + 1e-6))  # Normalize uncertainty

        return mean_prediction, uncertainty

# ✅ Instantiate Bayesian GPT-4 Model
bayesian_model = BayesianGPT4()

# ✅ Function: Generate Response Without Uncertainty
def generate_response_no_uncertainty(input_text):
    """Generate response using GPT-4 without uncertainty modeling."""
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": input_text}],
        max_tokens=50,
        temperature=1.0
    )
    return response.choices[0].message.content

# ✅ Clinical Uncertainty Score (CUS)
def compute_cus(uncertainties):
    """Compute the Clinical Uncertainty Score (CUS)."""
    return np.mean(uncertainties)

# ✅ Improved Zero-Shot Trustworthiness Index (ZTI)
def compute_zti(cus_scores, epsilon=1e-6):
    """Compute Zero-Shot Trustworthiness Index (ZTI)."""
    if len(cus_scores) < 2:
        return 1.0  # Default to fully trustworthy if only one sample is available

    mean_cus = np.mean(cus_scores)
    std_cus = np.std(cus_scores)
    max_cus = max(np.max(cus_scores), epsilon)  # Ensure non-zero max for stability

    zti = 1 - (np.log1p(abs(mean_cus - std_cus)) / np.log1p(max_cus))  # Logarithmic scaling
    return max(0, min(zti, 1))  # Ensure ZTI stays within [0,1] range

# ✅ Compute CUS and ZTI across multiple samples
cus_scores_no_uncertainty = []
cus_scores_with_uncertainty = []

for i in range(num_samples):
    # 🔹 Without Uncertainty
    response_no_uncertainty = generate_response_no_uncertainty(questions[i])
    print(f"Sample {i+1} - GPT-4 Response (No Uncertainty): {response_no_uncertainty}")

    cus_score_no_uncertainty = len(response_no_uncertainty.split())  # Length-based uncertainty proxy
    cus_scores_no_uncertainty.append(cus_score_no_uncertainty)
    print(f"Sample {i+1} - CUS (No Uncertainty): {cus_score_no_uncertainty}")

    # 🔹 With Bayesian Uncertainty Modeling
    mean_pred, uncertainty = bayesian_model(questions[i], n_samples=10)
    cus_score = compute_cus([uncertainty])
    cus_scores_with_uncertainty.append(cus_score)
    print(f"Sample {i+1} - CUS (With Uncertainty): {cus_score}")

# ✅ Compute Overall ZTI
zti_score_no_uncertainty = compute_zti(cus_scores_no_uncertainty)
zti_score_with_uncertainty = compute_zti(cus_scores_with_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) No Uncertainty:", zti_score_no_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) With Uncertainty:", zti_score_with_uncertainty)

# ✅ Generate Final Comparison Report
comparison_results = {
    "CUS_No_Uncertainty": cus_scores_no_uncertainty,
    "CUS_With_Uncertainty": cus_scores_with_uncertainty,
    "ZTI_No_Uncertainty": zti_score_no_uncertainty,
    "ZTI_With_Uncertainty": zti_score_with_uncertainty,
}
print("Comparison Results:", comparison_results)


Sample 1 - GPT-4 Response (No Uncertainty): Yes, mitochondria play a crucial role in the remodelling of lace plant leaves during programmed cell death. They are involved in energy production and the regulation of cellular metabolism, which are essential in the process of programmed cell death or apoptosis. Additionally,
Sample 1 - CUS (No Uncertainty): 42
Sample 1 - CUS (With Uncertainty): 0.1333048003002794
Sample 2 - GPT-4 Response (No Uncertainty): Strabismus and amblyopia are two eye disorders that are usually diagnosed and treated in childhood. Strabismus is a disorder in which the eyes are misaligned and point in different directions, while amblyopia, often referred to as
Sample 2 - CUS (No Uncertainty): 36
Sample 2 - CUS (With Uncertainty): 0.08783823281498368
Sample 3 - GPT-4 Response (No Uncertainty): Syncope, or fainting, is a condition characterized by a temporary loss of consciousness usually caused by insufficient blood flow to the brain. While it can occur in infants, it'

# GPT-4o on PubMedQA

In [ ]:
import openai
import numpy as np
import json
import time
import requests
import torch
import torch.nn as nn


# ✅ Load dataset from GitHub (PubMedQA)
dataset_url = "https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json"
response = requests.get(dataset_url)
data = response.json()

# ✅ Extract questions
questions = [item["QUESTION"] for item in data.values()]
num_samples = min(10, len(questions))  # Limit to 10 samples for evaluation

# ✅ Define BayesianGPT-4o Model for Uncertainty Estimation
class BayesianGPT4o(nn.Module):
    def __init__(self):
        super(BayesianGPT4o, self).__init__()
        self.dropout = nn.Dropout(p=0.3)  # Apply MC Dropout (simulated via multiple API calls)

    def forward(self, input_text, n_samples=10):
        """Simulate Bayesian inference with multiple API calls (Monte Carlo Estimation)."""
        predictions = []

        for _ in range(n_samples):
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[{"role": "user", "content": input_text}],
                max_tokens=50,
                temperature=np.random.uniform(0.7, 1.2)  # Randomized temp for diversity
            )
            predictions.append(response.choices[0].message.content)
            time.sleep(1)  # Avoid rate limit

        # Convert predictions to numerical values
        prediction_lengths = [len(pred.split()) for pred in predictions]
        mean_prediction = np.mean(prediction_lengths)
        uncertainty = np.abs(np.std(prediction_lengths) / (mean_prediction + 1e-6))  # Normalize uncertainty

        return mean_prediction, uncertainty

# ✅ Instantiate Bayesian GPT-4o Model
bayesian_model = BayesianGPT4o()

# ✅ Function: Generate Response Without Uncertainty
def generate_response_no_uncertainty(input_text):
    """Generate response using GPT-4o without uncertainty modeling."""
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": input_text}],
        max_tokens=50,
        temperature=1.0
    )
    return response.choices[0].message.content

# ✅ Clinical Uncertainty Score (CUS)
def compute_cus(uncertainties):
    """Compute the Clinical Uncertainty Score (CUS)."""
    return np.mean(uncertainties)

# ✅ Improved Zero-Shot Trustworthiness Index (ZTI)
def compute_zti(cus_scores, epsilon=1e-6):
    """Compute Zero-Shot Trustworthiness Index (ZTI)."""
    if len(cus_scores) < 2:
        return 1.0  # Default to fully trustworthy if only one sample is available

    mean_cus = np.mean(cus_scores)
    std_cus = np.std(cus_scores)
    max_cus = max(np.max(cus_scores), epsilon)  # Ensure non-zero max for stability

    zti = 1 - (np.log1p(abs(mean_cus - std_cus)) / np.log1p(max_cus))  # Logarithmic scaling
    return max(0, min(zti, 1))  # Ensure ZTI stays within [0,1] range

# ✅ Compute CUS and ZTI across multiple samples
cus_scores_no_uncertainty = []
cus_scores_with_uncertainty = []

for i in range(num_samples):
    # 🔹 Without Uncertainty
    response_no_uncertainty = generate_response_no_uncertainty(questions[i])
    print(f"Sample {i+1} - GPT-4o Response (No Uncertainty): {response_no_uncertainty}")

    cus_score_no_uncertainty = len(response_no_uncertainty.split())  # Length-based uncertainty proxy
    cus_scores_no_uncertainty.append(cus_score_no_uncertainty)
    print(f"Sample {i+1} - CUS (No Uncertainty): {cus_score_no_uncertainty}")

    # 🔹 With Bayesian Uncertainty Modeling
    mean_pred, uncertainty = bayesian_model(questions[i], n_samples=10)
    cus_score = compute_cus([uncertainty])
    cus_scores_with_uncertainty.append(cus_score)
    print(f"Sample {i+1} - CUS (With Uncertainty): {cus_score}")

# ✅ Compute Overall ZTI
zti_score_no_uncertainty = compute_zti(cus_scores_no_uncertainty)
zti_score_with_uncertainty = compute_zti(cus_scores_with_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) No Uncertainty:", zti_score_no_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) With Uncertainty:", zti_score_with_uncertainty)

# ✅ Generate Final Comparison Report
comparison_results = {
    "CUS_No_Uncertainty": cus_scores_no_uncertainty,
    "CUS_With_Uncertainty": cus_scores_with_uncertainty,
    "ZTI_No_Uncertainty": zti_score_no_uncertainty,
    "ZTI_With_Uncertainty": zti_score_with_uncertainty,
}
print("Comparison Results:", comparison_results)


Sample 1 - GPT-4o Response (No Uncertainty): Yes, mitochondria play a role in the programmed cell death (PCD) associated with the remodeling of lace plant leaves. The lace plant (Aponogeton madagascariensis) is known for its striking pattern of perforated leaves,
Sample 1 - CUS (No Uncertainty): 33
Sample 1 - CUS (With Uncertainty): 0.02549063635138884
Sample 2 - GPT-4o Response (No Uncertainty): Landolt C and Snellen E charts are both commonly used tools in assessing visual acuity, but they have different characteristics that may affect their use in evaluating conditions like strabismus and amblyopia.

1. **Landolt C Chart**:

Sample 2 - CUS (No Uncertainty): 36
Sample 2 - CUS (With Uncertainty): 0.04812834095913528
Sample 3 - GPT-4o Response (No Uncertainty): Syncope during bathing in infants is a concerning symptom and not typically classified as a form of water-induced urticaria. Water-induced urticaria, also known as aquagenic urticaria, is a rare condition where contact with wate

# GPT 4.5 on PubMedQA

In [ ]:
import openai
import numpy as np
import json
import time
import requests
import torch
import torch.nn as nn
from openai import OpenAI

# ✅ Load dataset from GitHub (PubMedQA)
dataset_url = "https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json"
response = requests.get(dataset_url)
data = response.json()

# ✅ Extract questions
questions = [item["QUESTION"] for item in data.values()]
num_samples = min(10, len(questions))  # Limit to 10 samples for evaluation

# ✅ Define OpenAI Model (Only GPT-4.5 Preview)
model_name = "gpt-4.5-preview"

# ✅ Define BayesianGPT for Uncertainty Estimation
class BayesianGPT(nn.Module):
    def __init__(self):
        super(BayesianGPT, self).__init__()
        self.dropout = nn.Dropout(p=0.3)  # Apply MC Dropout (simulated via multiple API calls)

    def forward(self, input_text, n_samples=10):
        """Simulate Bayesian inference with multiple API calls (Monte Carlo Estimation)."""
        predictions = []

        for _ in range(n_samples):
            response = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": input_text}],
                max_tokens=50,
                temperature=np.random.uniform(0.7, 1.2)  # Randomized temp for diversity
            )
            predictions.append(response.choices[0].message.content)
            time.sleep(1)  # Avoid rate limit

        # Convert predictions to numerical values
        prediction_lengths = [len(pred.split()) for pred in predictions]
        mean_prediction = np.mean(prediction_lengths)
        uncertainty = np.abs(np.std(prediction_lengths) / (mean_prediction + 1e-6))  # Normalize uncertainty

        return mean_prediction, uncertainty

# ✅ Instantiate Bayesian GPT Model
bayesian_model = BayesianGPT()

# ✅ Function: Generate Response Without Uncertainty
def generate_response_no_uncertainty(input_text):
    """Generate response using GPT-4.5 Preview without uncertainty modeling."""
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": input_text}],
        max_tokens=50,
        temperature=1.0
    )
    return response.choices[0].message.content

# ✅ Clinical Uncertainty Score (CUS)
def compute_cus(uncertainties):
    """Compute the Clinical Uncertainty Score (CUS)."""
    return np.mean(uncertainties)

# ✅ Improved Zero-Shot Trustworthiness Index (ZTI)
def compute_zti(cus_scores, epsilon=1e-6):
    """Compute Zero-Shot Trustworthiness Index (ZTI)."""
    if len(cus_scores) < 2:
        return 1.0  # Default to fully trustworthy if only one sample is available

    mean_cus = np.mean(cus_scores)
    std_cus = np.std(cus_scores)
    max_cus = max(np.max(cus_scores), epsilon)  # Ensure non-zero max for stability

    zti = 1 - (np.log1p(abs(mean_cus - std_cus)) / np.log1p(max_cus))  # Logarithmic scaling
    return max(0, min(zti, 1))  # Ensure ZTI stays within [0,1] range

# ✅ Compute CUS and ZTI across multiple samples
cus_scores_no_uncertainty = []
cus_scores_with_uncertainty = []

for i in range(num_samples):
    # 🔹 Without Uncertainty
    response_no_uncertainty = generate_response_no_uncertainty(questions[i])
    print(f"Sample {i+1} - GPT-4.5 Preview Response (No Uncertainty): {response_no_uncertainty}")

    cus_score_no_uncertainty = len(response_no_uncertainty.split())  # Length-based uncertainty proxy
    cus_scores_no_uncertainty.append(cus_score_no_uncertainty)
    print(f"Sample {i+1} - CUS (No Uncertainty): {cus_score_no_uncertainty}")

    # 🔹 With Bayesian Uncertainty Modeling
    mean_pred, uncertainty = bayesian_model(questions[i], n_samples=10)
    cus_score = compute_cus([uncertainty])
    cus_scores_with_uncertainty.append(cus_score)
    print(f"Sample {i+1} - CUS (With Uncertainty): {cus_score}")

# ✅ Compute Overall ZTI
zti_score_no_uncertainty = compute_zti(cus_scores_no_uncertainty)
zti_score_with_uncertainty = compute_zti(cus_scores_with_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) No Uncertainty:", zti_score_no_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) With Uncertainty:", zti_score_with_uncertainty)

# ✅ Generate Final Comparison Report
comparison_results = {
    "Model": "GPT-4.5 Preview",
    "CUS_No_Uncertainty": cus_scores_no_uncertainty,
    "CUS_With_Uncertainty": cus_scores_with_uncertainty,
    "ZTI_No_Uncertainty": zti_score_no_uncertainty,
    "ZTI_With_Uncertainty": zti_score_with_uncertainty,
}
print("Comparison Results:", comparison_results)


Sample 1 - GPT-4.5 Preview Response (No Uncertainty): Yes. In lace plants (*Aponogeton madagascariensis*), mitochondria play a crucial role during leaf remodeling through programmed cell death (PCD). Lace plants prominently exhibit programmed cell death, creating their characteristic lace-like leaf perforations
Sample 1 - CUS (No Uncertainty): 32
Sample 1 - CUS (With Uncertainty): 0.03922339257970783
Sample 2 - GPT-4.5 Preview Response (No Uncertainty): Landolt C and Snellen E tests are both widely utilized visual acuity tests, but subtle differences may emerge when assessing patients with strabismic amblyopia. Strabismic amblyopia, unlike refractive or anisometropic amblyopia
Sample 2 - CUS (No Uncertainty): 31
Sample 2 - CUS (With Uncertainty): 0.06395761754816835
Sample 3 - GPT-4.5 Preview Response (No Uncertainty): Syncope during bathing in infants may sometimes represent a pediatric form of water-induced urticaria. Water-induced urticaria, also known as aquagenic urticaria, is a r

# GPT Neo-1.3B on PubMedQA

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np
import requests
from scipy.special import softmax

# ✅ Load dataset from GitHub (PubMedQA)
dataset_url = "https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json"
response = requests.get(dataset_url)
data = response.json()

# ✅ Extract questions
questions = [item["QUESTION"] for item in data.values()]
num_samples = min(10, len(questions))  # Limit to 10 samples for evaluation

# ✅ Load GPT-Neo 1.3B Model & Tokenizer
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-1.3B")
tokenizer.pad_token = tokenizer.eos_token  # Explicitly set padding token
model = AutoModelForCausalLM.from_pretrained("EleutherAI/gpt-neo-1.3B")

# ✅ Function: Generate Response Without Uncertainty
def generate_response_no_uncertainty(input_text):
    """Generate response using GPT-Neo 1.3B without uncertainty modeling."""
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    output_sequences = model.generate(inputs["input_ids"], max_new_tokens=50, pad_token_id=tokenizer.eos_token_id)
    response = tokenizer.decode(output_sequences[0], skip_special_tokens=True)
    return response

# ✅ Bayesian Calibration using Monte Carlo Dropout
class BayesianGPTNeo(nn.Module):
    def __init__(self, base_model):
        super(BayesianGPTNeo, self).__init__()
        self.base_model = base_model
        self.dropout = nn.Dropout(p=0.3)  # Apply MC Dropout for Bayesian Estimation

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids, attention_mask=attention_mask)
        logits = self.dropout(outputs.logits)  # Apply dropout multiple times during inference
        return logits

# ✅ Apply Bayesian Model
bayesian_model = BayesianGPTNeo(model)
bayesian_model.train()

# ✅ Function: Estimate Uncertainty with Monte Carlo Dropout
def estimate_uncertainty_gpt(input_text, n_samples=20):
    """Estimate uncertainty using Bayesian Monte Carlo Dropout."""
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    input_ids, attention_mask = inputs["input_ids"], inputs["attention_mask"]
    predictions = []

    for _ in range(n_samples):
        with torch.no_grad():
            logits = bayesian_model(input_ids, attention_mask)
            predictions.append(softmax(logits.numpy(), axis=-1))  # Normalize logits using softmax

    predictions = np.array(predictions)
    mean_prediction = predictions.mean(axis=0)
    uncertainty = np.abs(predictions.std(axis=0) / (mean_prediction + 1e-6))  # Normalized uncertainty

    return mean_prediction, uncertainty

# ✅ Clinical Uncertainty Score (CUS)
def compute_cus(uncertainties):
    """Compute the Clinical Uncertainty Score (CUS)."""
    return np.mean(uncertainties)

# ✅ Improved Zero-Shot Trustworthiness Index (ZTI)
def compute_zti(cus_scores, epsilon=1e-6):
    """Compute Zero-Shot Trustworthiness Index (ZTI)."""
    if len(cus_scores) < 2:
        return 1.0  # Default to fully trustworthy if only one sample is available

    mean_cus = np.mean(cus_scores)
    std_cus = np.std(cus_scores)
    max_cus = max(np.max(cus_scores), epsilon)  # Ensure non-zero max for stability

    zti = 1 - (np.log1p(abs(mean_cus - std_cus)) / np.log1p(max_cus))  # Ensure non-negative log values
    return max(0, min(zti, 1))  # Ensure ZTI stays within [0,1] range

# ✅ Compute CUS and ZTI across multiple samples
cus_scores_no_uncertainty = []
cus_scores_with_uncertainty = []

for i in range(num_samples):
    # 🔹 Without Uncertainty
    response_no_uncertainty = generate_response_no_uncertainty(questions[i])
    print(f"Sample {i+1} - GPT-Neo Response (No Uncertainty): {response_no_uncertainty}")

    cus_score_no_uncertainty = len(response_no_uncertainty.split())  # Length-based uncertainty proxy
    cus_scores_no_uncertainty.append(cus_score_no_uncertainty)
    print(f"Sample {i+1} - CUS (No Uncertainty): {cus_score_no_uncertainty}")

    # 🔹 With Bayesian Uncertainty Modeling
    _, uncertainty = estimate_uncertainty_gpt(questions[i])
    cus_score = compute_cus(uncertainty)
    cus_scores_with_uncertainty.append(cus_score)
    print(f"Sample {i+1} - CUS (With Uncertainty): {cus_score}")

# ✅ Compute Overall ZTI
zti_score_no_uncertainty = compute_zti(cus_scores_no_uncertainty)
zti_score_with_uncertainty = compute_zti(cus_scores_with_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) No Uncertainty:", zti_score_no_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) With Uncertainty:", zti_score_with_uncertainty)

# ✅ Generate Final Comparison Report
comparison_results = {
    "CUS_No_Uncertainty": cus_scores_no_uncertainty,
    "CUS_With_Uncertainty": cus_scores_with_uncertainty,
    "ZTI_No_Uncertainty": zti_score_no_uncertainty,
    "ZTI_With_Uncertainty": zti_score_with_uncertainty,
}
print("Comparison Results:", comparison_results)


Sample 1 - GPT-Neo Response (No Uncertainty): Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?

Mitochondria are the powerhouses of the cell, and they are involved in many processes, including energy production, apoptosis, and programmed cell death. Mitochondria are also involved in the regulation of plant development and growth.


Sample 1 - CUS (No Uncertainty): 48
Sample 1 - CUS (With Uncertainty): 1.5274032354354858
Sample 2 - GPT-Neo Response (No Uncertainty): Landolt C and snellen e acuity: differences in strabismus amblyopia?

Abstract

Objective: To investigate the relationship between strabismus amblyopia and visual acuity (VA) in children with strabismus.

Methods: A total of 5,890 children with strab
Sample 2 - CUS (No Uncertainty): 35
Sample 2 - CUS (With Uncertainty): 1.526963233947754
Sample 3 - GPT-Neo Response (No Uncertainty): Syncope during bathing in infants, a pediatric form of water-induced urticaria?
To report a case of wa

# BERT Base on PubMedQA

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np
import requests
from scipy.special import softmax

# ✅ Load dataset from GitHub (PubMedQA)
dataset_url = "https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json"
response = requests.get(dataset_url)
data = response.json()

# ✅ Extract questions
questions = [item["QUESTION"] for item in data.values()]
num_samples = min(10, len(questions))  # Limit to 10 samples for evaluation

# ✅ Load BERT Model & Tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)

# ✅ Function: Predict Without Uncertainty
def predict_no_uncertainty(input_text):
    """Generate prediction using BERT without uncertainty modeling."""
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits
    probs = softmax(logits.numpy(), axis=1)
    return probs

# ✅ Bayesian Calibration using Monte Carlo Dropout
class BayesianBERT(nn.Module):
    def __init__(self, base_model):
        super(BayesianBERT, self).__init__()
        self.base_model = base_model
        self.dropout = nn.Dropout(p=0.3)  # MC Dropout for Bayesian Estimation

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids, attention_mask=attention_mask)
        logits = self.dropout(outputs.logits)  # Apply dropout multiple times during inference
        return logits

# ✅ Apply Bayesian Model
bayesian_model = BayesianBERT(model)
bayesian_model.train()

# ✅ Function: Estimate Uncertainty with Monte Carlo Dropout
def estimate_uncertainty_bert(input_text, n_samples=20):
    """Estimate uncertainty using Bayesian Monte Carlo Dropout."""
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    input_ids, attention_mask = inputs["input_ids"], inputs["attention_mask"]
    predictions = []

    for _ in range(n_samples):
        with torch.no_grad():
            logits = bayesian_model(input_ids, attention_mask)
            predictions.append(softmax(logits.numpy(), axis=1))  # Normalize logits using softmax

    predictions = np.array(predictions)
    mean_prediction = predictions.mean(axis=0)
    uncertainty = np.abs(predictions.std(axis=0) / (mean_prediction + 1e-6))  # Normalized uncertainty

    return mean_prediction, uncertainty

# ✅ Clinical Uncertainty Score (CUS)
def compute_cus(uncertainties):
    """Compute the Clinical Uncertainty Score (CUS)."""
    return np.mean(uncertainties)

# ✅ Improved Zero-Shot Trustworthiness Index (ZTI)
def compute_zti(cus_scores, epsilon=1e-6):
    """Compute Zero-Shot Trustworthiness Index (ZTI)."""
    if len(cus_scores) < 2:
        return 1.0  # Default to fully trustworthy if only one sample is available

    mean_cus = np.mean(cus_scores)
    std_cus = np.std(cus_scores)
    max_cus = max(np.max(cus_scores), epsilon)  # Ensure non-zero max for stability

    zti = 1 - (np.log1p(abs(mean_cus - std_cus)) / np.log1p(max_cus))  # Ensure non-negative log values
    return max(0, min(zti, 1))  # Ensure ZTI stays within [0,1] range

# ✅ Compute CUS and ZTI across multiple samples
cus_scores_no_uncertainty = []
cus_scores_with_uncertainty = []

for i in range(num_samples):
    # 🔹 Without Uncertainty
    probs_no_uncertainty = predict_no_uncertainty(questions[i])
    cus_score_no_uncertainty = compute_cus(probs_no_uncertainty)
    cus_scores_no_uncertainty.append(cus_score_no_uncertainty)
    print(f"Sample {i+1} - CUS (No Uncertainty): {cus_score_no_uncertainty}")

    # 🔹 With Bayesian Uncertainty Modeling
    _, uncertainty = estimate_uncertainty_bert(questions[i])
    cus_score = compute_cus(uncertainty)
    cus_scores_with_uncertainty.append(cus_score)
    print(f"Sample {i+1} - CUS (With Uncertainty): {cus_score}")

# ✅ Compute Overall ZTI
zti_score_no_uncertainty = compute_zti(cus_scores_no_uncertainty)
zti_score_with_uncertainty = compute_zti(cus_scores_with_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) No Uncertainty:", zti_score_no_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) With Uncertainty:", zti_score_with_uncertainty)

# ✅ Generate Final Comparison Report
comparison_results = {
    "CUS_No_Uncertainty": cus_scores_no_uncertainty,
    "CUS_With_Uncertainty": cus_scores_with_uncertainty,
    "ZTI_No_Uncertainty": zti_score_no_uncertainty,
    "ZTI_With_Uncertainty": zti_score_with_uncertainty,
}
print("Comparison Results:", comparison_results)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Sample 1 - CUS (No Uncertainty): 0.25
Sample 1 - CUS (With Uncertainty): 0.19944503903388977
Sample 2 - CUS (No Uncertainty): 0.25
Sample 2 - CUS (With Uncertainty): 0.2011132389307022
Sample 3 - CUS (No Uncertainty): 0.2499999850988388
Sample 3 - CUS (With Uncertainty): 0.19842314720153809
Sample 4 - CUS (No Uncertainty): 0.25
Sample 4 - CUS (With Uncertainty): 0.2093600630760193
Sample 5 - CUS (No Uncertainty): 0.25
Sample 5 - CUS (With Uncertainty): 0.22263461351394653
Sample 6 - CUS (No Uncertainty): 0.25
Sample 6 - CUS (With Uncertainty): 0.204816535115242
Sample 7 - CUS (No Uncertainty): 0.2500000298023224
Sample 7 - CUS (With Uncertainty): 0.23612436652183533
Sample 8 - CUS (No Uncertainty): 0.2499999850988388
Sample 8 - CUS (With Uncertainty): 0.20275470614433289
Sample 9 - CUS (No Uncertainty): 0.25
Sample 9 - CUS (With Uncertainty): 0.2282758504152298
Sample 10 - CUS (No Uncertainty): 0.2499999701976776
Sample 10 - CUS (With Uncertainty): 0.22415263950824738
Zero-Shot Trustwo

# Other Variants of BERT Models on PubMedQA

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import requests
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax

# ✅ Load dataset from GitHub (PubMedQA)
dataset_url = "https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json"
response = requests.get(dataset_url)
data = response.json()

# ✅ Extract questions
questions = [item["QUESTION"] for item in data.values()]
num_samples = min(10, len(questions))  # Evaluate on 10 questions for better stability

# ✅ Define model names for different BERT variants
BERT_MODELS = {
    "BERT-Large": "bert-large-uncased",
    "BioBERT": "dmis-lab/biobert-base-cased-v1.1",
    "ClinicalBERT": "emilyalsentzer/Bio_ClinicalBERT",
    "RoBERTa": "roberta-base"
}

# ✅ Bayesian Calibration using Monte Carlo Dropout
class BayesianBERT(nn.Module):
    def __init__(self, base_model):
        super(BayesianBERT, self).__init__()
        self.base_model = base_model
        self.dropout = nn.Dropout(p=0.3)  # MC Dropout for Bayesian Estimation

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids, attention_mask=attention_mask)
        logits = self.dropout(outputs.logits)  # Apply dropout multiple times during inference
        return logits

# ✅ Function: Predict Without Uncertainty
def predict_no_uncertainty(model, tokenizer, input_text):
    """Generate prediction using BERT without uncertainty modeling."""
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits
    probs = softmax(logits.numpy(), axis=1)
    return probs

# ✅ Function: Estimate Uncertainty with Monte Carlo Dropout
def estimate_uncertainty_bert(model, tokenizer, input_text, n_samples=20):
    """Estimate uncertainty using Bayesian Monte Carlo Dropout."""
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    input_ids, attention_mask = inputs["input_ids"], inputs["attention_mask"]
    predictions = []

    for _ in range(n_samples):
        with torch.no_grad():
            logits = model(input_ids, attention_mask)
            predictions.append(softmax(logits.numpy(), axis=1))  # Normalize logits using softmax

    predictions = np.array(predictions)
    mean_prediction = predictions.mean(axis=0)
    uncertainty = np.abs(predictions.std(axis=0) / (mean_prediction + 1e-6))  # Normalized uncertainty

    return mean_prediction, uncertainty

# ✅ Clinical Uncertainty Score (CUS)
def compute_cus(uncertainties):
    """Compute the Clinical Uncertainty Score (CUS)."""
    return np.mean(uncertainties)

# ✅ Improved Zero-Shot Trustworthiness Index (ZTI)
def compute_zti(cus_scores, epsilon=1e-6):
    """Compute Zero-Shot Trustworthiness Index (ZTI)."""
    if len(cus_scores) < 2:
        return 1.0  # Default to fully trustworthy if only one sample is available

    mean_cus = np.mean(cus_scores)
    std_cus = np.std(cus_scores)
    max_cus = max(np.max(cus_scores), epsilon)  # Ensure non-zero max for stability

    zti = 1 - (np.log1p(abs(mean_cus - std_cus)) / np.log1p(max_cus))  # Ensure non-negative log values
    return max(0, min(zti, 1))  # Ensure ZTI stays within [0,1] range

# ✅ Iterate over different BERT models
results = {}

for model_name, model_path in BERT_MODELS.items():
    print(f"\n====== Running {model_name} on PubMedQA ======")

    # Load pre-trained model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=4)

    # Apply Bayesian Model
    bayesian_model = BayesianBERT(model)
    bayesian_model.train()

    cus_scores_no_uncertainty = []
    cus_scores_with_uncertainty = []

    for i in range(num_samples):
        # 🔹 Without Uncertainty
        probs_no_uncertainty = predict_no_uncertainty(model, tokenizer, questions[i])
        cus_score_no_uncertainty = compute_cus(probs_no_uncertainty)
        cus_scores_no_uncertainty.append(cus_score_no_uncertainty)
        print(f"Sample {i+1} - {model_name} CUS (No Uncertainty): {cus_score_no_uncertainty}")

        # 🔹 With Bayesian Uncertainty Modeling
        _, uncertainty = estimate_uncertainty_bert(bayesian_model, tokenizer, questions[i])
        cus_score = compute_cus(uncertainty)
        cus_scores_with_uncertainty.append(cus_score)
        print(f"Sample {i+1} - {model_name} CUS (With Uncertainty): {cus_score}")

    # ✅ Compute Overall ZTI
    zti_score_no_uncertainty = compute_zti(cus_scores_no_uncertainty)
    zti_score_with_uncertainty = compute_zti(cus_scores_with_uncertainty)
    print(f"{model_name} Zero-Shot Trustworthiness Index (ZTI) No Uncertainty:", zti_score_no_uncertainty)
    print(f"{model_name} Zero-Shot Trustworthiness Index (ZTI) With Uncertainty:", zti_score_with_uncertainty)

    # ✅ Store results
    results[model_name] = {
        "CUS_No_Uncertainty": cus_scores_no_uncertainty,
        "CUS_With_Uncertainty": cus_scores_with_uncertainty,
        "ZTI_No_Uncertainty": zti_score_no_uncertainty,
        "ZTI_With_Uncertainty": zti_score_with_uncertainty,
    }

# ✅ Print Final Comparison Report
print("\n===== Final Comparison Report =====")
for model_name, metrics in results.items():
    print(f"\nModel: {model_name}")
    print("CUS (No Uncertainty):", metrics["CUS_No_Uncertainty"])
    print("CUS (With Uncertainty):", metrics["CUS_With_Uncertainty"])
    print("ZTI (No Uncertainty):", metrics["ZTI_No_Uncertainty"])
    print("ZTI (With Uncertainty):", metrics["ZTI_With_Uncertainty"])



====== Running BERT-Large on PubMedQA ======


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-large-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Sample 1 - BERT-Large CUS (No Uncertainty): 0.25
Sample 1 - BERT-Large CUS (With Uncertainty): 0.2536579370498657
Sample 2 - BERT-Large CUS (No Uncertainty): 0.25
Sample 2 - BERT-Large CUS (With Uncertainty): 0.37691259384155273
Sample 3 - BERT-Large CUS (No Uncertainty): 0.25
Sample 3 - BERT-Large CUS (With Uncertainty): 0.2758502662181854
Sample 4 - BERT-Large CUS (No Uncertainty): 0.25
Sample 4 - BERT-Large CUS (With Uncertainty): 0.2764850854873657
Sample 5 - BERT-Large CUS (No Uncertainty): 0.25
Sample 5 - BERT-Large CUS (With Uncertainty): 0.32851800322532654
Sample 6 - BERT-Large CUS (No Uncertainty): 0.25
Sample 6 - BERT-Large CUS (With Uncertainty): 0.30346232652664185
Sample 7 - BERT-Large CUS (No Uncertainty): 0.25
Sample 7 - BERT-Large CUS (With Uncertainty): 0.24843698740005493
Sample 8 - BERT-Large CUS (No Uncertainty): 0.25
Sample 8 - BERT-Large CUS (With Uncertainty): 0.32731810212135315
Sample 9 - BERT-Large CUS (No Uncertainty): 0.2499999850988388
Sample 9 - BERT-Larg

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

... (15 outputs left out)

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Sample 1 - RoBERTa CUS (No Uncertainty): 0.25
Sample 1 - RoBERTa CUS (With Uncertainty): 0.13272657990455627
Sample 2 - RoBERTa CUS (No Uncertainty): 0.2500000298023224
Sample 2 - RoBERTa CUS (With Uncertainty): 0.10878384113311768
Sample 3 - RoBERTa CUS (No Uncertainty): 0.25
Sample 3 - RoBERTa CUS (With Uncertainty): 0.12835603952407837
Sample 4 - RoBERTa CUS (No Uncertainty): 0.25
Sample 4 - RoBERTa CUS (With Uncertainty): 0.09496794641017914
Sample 5 - RoBERTa CUS (No Uncertainty): 0.25
Sample 5 - RoBERTa CUS (With Uncertainty): 0.10366171598434448
Sample 6 - RoBERTa CUS (No Uncertainty): 0.2499999850988388
Sample 6 - RoBERTa CUS (With Uncertainty): 0.14310190081596375
Sample 7 - RoBERTa CUS (No Uncertainty): 0.25
Sample 7 - RoBERTa CUS (With Uncertainty): 0.13000506162643433
Sample 8 - RoBERTa CUS (No Uncertainty): 0.25
Sample 8 - RoBERTa CUS (With Uncertainty): 0.10194769501686096
Sample 9 - RoBERTa CUS (No Uncertainty): 0.25
Sample 9 - RoBERTa CUS (With Uncertainty): 0.116855248

# LLaMA-2 7B on PubMedQA

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import requests
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.special import softmax

# ✅ Load dataset from GitHub (PubMedQA)
dataset_url = "https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json"
response = requests.get(dataset_url)
data = response.json()

# ✅ Extract questions
questions = [item["QUESTION"] for item in data.values()]
num_samples = min(10, len(questions))  # Evaluate on 10 questions for better stability

# ✅ Load LLaMA-2 7B model
MODEL_NAME = "meta-llama/Llama-2-7b-hf"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # Fix: Set padding token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")

# ✅ Enable dropout for Bayesian Approximation (Monte Carlo)
class BayesianLLaMA(nn.Module):
    def __init__(self, base_model):
        super(BayesianLLaMA, self).__init__()
        self.base_model = base_model
        self.dropout = nn.Dropout(p=0.3)  # Apply MC Dropout

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        logits = self.dropout(outputs.logits)  # Apply dropout at logits level
        return logits

# ✅ Wrap the model with Bayesian uncertainty modeling
bayesian_model = BayesianLLaMA(model)
bayesian_model.train()  # Ensure dropout is active

# ✅ Function: Generate Response Without Uncertainty
def generate_response_no_uncertainty(input_text):
    """Generate response using LLaMA-2 without uncertainty modeling."""
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512, pad_to_multiple_of=8).to("cuda")
    with torch.no_grad():
        output_sequences = model.generate(inputs["input_ids"], max_new_tokens=50, pad_token_id=tokenizer.pad_token_id)
    response = tokenizer.decode(output_sequences[0], skip_special_tokens=True)
    return response

# ✅ Function: Bayesian Uncertainty Estimation
def estimate_uncertainty_llama(input_text, n_samples=10):
    """Estimate uncertainty using Bayesian Monte Carlo Dropout."""
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512, pad_to_multiple_of=8).to("cuda")
    predictions = []

    for _ in range(n_samples):
        with torch.no_grad():
            logits = bayesian_model(inputs["input_ids"], inputs["attention_mask"])
            predictions.append(logits.cpu().numpy())

    predictions = np.array(predictions)
    mean_prediction = predictions.mean(axis=0)
    uncertainty = np.abs(predictions.std(axis=0) / (mean_prediction + 1e-6))  # Normalize uncertainty

    return mean_prediction, uncertainty

# ✅ Clinical Uncertainty Score (CUS)
def compute_cus(uncertainties):
    """Compute Clinical Uncertainty Score (CUS)."""
    return np.mean(uncertainties)

# ✅ Zero-Shot Trustworthiness Index (ZTI)
def compute_zti(cus_scores, epsilon=1e-6):
    """Compute Zero-Shot Trustworthiness Index (ZTI)."""
    if len(cus_scores) < 2:
        return 1.0  # Default to fully trustworthy if only one sample is available

    mean_cus = np.mean(cus_scores)
    std_cus = np.std(cus_scores)
    max_cus = max(np.max(cus_scores), epsilon)  # Ensure non-zero max for stability

    zti = 1 - (np.log1p(abs(mean_cus - std_cus)) / np.log1p(max_cus))  # Logarithmic scaling
    return max(0, min(zti, 1))  # Ensure ZTI stays within [0,1] range

# ✅ Compute CUS and ZTI across multiple samples
cus_scores_no_uncertainty = []
cus_scores_with_uncertainty = []

for i in range(num_samples):
    # 🔹 Without Uncertainty
    response_no_uncertainty = generate_response_no_uncertainty(questions[i])
    print(f"Sample {i+1} - LLaMA-2 7B Response (No Uncertainty): {response_no_uncertainty}")
    cus_score_no_uncertainty = len(response_no_uncertainty.split())  # Length-based proxy
    cus_scores_no_uncertainty.append(cus_score_no_uncertainty)
    print(f"Sample {i+1} - CUS (No Uncertainty): {cus_score_no_uncertainty}")

    # 🔹 With Bayesian uncertainty modeling
    _, uncertainty = estimate_uncertainty_llama(questions[i])
    cus_score = compute_cus(uncertainty)
    cus_scores_with_uncertainty.append(cus_score)
    print(f"Sample {i+1} - CUS (With Uncertainty): {cus_score}")

# ✅ Compute ZTI scores
zti_score_no_uncertainty = compute_zti(cus_scores_no_uncertainty)
zti_score_with_uncertainty = compute_zti(cus_scores_with_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) No Uncertainty:", zti_score_no_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) With Uncertainty:", zti_score_with_uncertainty)

# ✅ Final Comparison Report
comparison_results = {
    "CUS_No_Uncertainty": cus_scores_no_uncertainty,
    "CUS_With_Uncertainty": cus_scores_with_uncertainty,
    "ZTI_No_Uncertainty": zti_score_no_uncertainty,
    "ZTI_With_Uncertainty": zti_score_with_uncertainty,
}
print("Comparison Results:", comparison_results)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

Sample 1 - LLaMA-2 7B Response (No Uncertainty): Do mitochondria play a role in remodelling lace plant leaves during programmed cell death? A 12-year-old boy was shot in the face in a drive-by shooting on the South Side early Monday, police said.
The boy was standing outside a home in the 700 block of West 1
Sample 1 - CUS (No Uncertainty): 48
Sample 1 - CUS (With Uncertainty): 0.66015625
Sample 2 - LLaMA-2 7B Response (No Uncertainty): Landolt C and snellen e acuity: differences in strabismus amblyopia? The Witcher 3: Wild Hunt (2015) - All Discussions - PlayStation 4
The Witcher 3: Wild Hunt (2015) - All Discussions

Sample 2 - CUS (No Uncertainty): 31
Sample 2 - CUS (With Uncertainty): 0.6591796875
Sample 3 - LLaMA-2 7B Response (No Uncertainty): Syncope during bathing in infants, a pediatric form of water-induced urticaria?DЊЋЪ
Fallout is a game created by the community that is a game that is a game.
Fallout is a game created by the community that is a game that is a game.
Fallout 

# LLaMA-2 13B

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import requests
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.special import softmax

# ✅ Load dataset from GitHub (PubMedQA)
dataset_url = "https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json"
response = requests.get(dataset_url)
data = response.json()

# ✅ Extract questions
questions = [item["QUESTION"] for item in data.values()]
num_samples = min(10, len(questions))  # Evaluate on 10 questions for better stability

# ✅ Load LLaMA-2 13B model
MODEL_NAME = "meta-llama/Llama-2-13b-hf"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # Fix padding token issue
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")

# ✅ Enable dropout for Bayesian Approximation (Monte Carlo)
class BayesianLLaMA(nn.Module):
    def __init__(self, base_model):
        super(BayesianLLaMA, self).__init__()
        self.base_model = base_model
        self.dropout = nn.Dropout(p=0.3)  # Apply MC Dropout

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        logits = self.dropout(outputs.logits)  # Apply dropout at logits level
        return logits

# ✅ Wrap the model with Bayesian uncertainty modeling
bayesian_model = BayesianLLaMA(model)
bayesian_model.train()  # Ensure dropout is active

# ✅ Function: Generate Response Without Uncertainty
def generate_response_no_uncertainty(input_text):
    """Generate response using LLaMA-2 13B without uncertainty modeling."""
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512, pad_to_multiple_of=8).to("cuda")
    with torch.no_grad():
        output_sequences = model.generate(inputs["input_ids"], max_new_tokens=50, pad_token_id=tokenizer.pad_token_id)
    response = tokenizer.decode(output_sequences[0], skip_special_tokens=True)
    return response

# ✅ Function: Bayesian Uncertainty Estimation
def estimate_uncertainty_llama(input_text, n_samples=10):
    """Estimate uncertainty using Bayesian Monte Carlo Dropout."""
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512, pad_to_multiple_of=8).to("cuda")
    predictions = []

    for _ in range(n_samples):
        with torch.no_grad():
            logits = bayesian_model(inputs["input_ids"], inputs["attention_mask"])
            predictions.append(logits.cpu().numpy())

    predictions = np.array(predictions)
    mean_prediction = predictions.mean(axis=0)
    uncertainty = np.abs(predictions.std(axis=0) / (mean_prediction + 1e-6))  # Normalize uncertainty

    return mean_prediction, uncertainty

# ✅ Clinical Uncertainty Score (CUS)
def compute_cus(uncertainties):
    """Compute Clinical Uncertainty Score (CUS)."""
    return np.mean(uncertainties)

# ✅ Zero-Shot Trustworthiness Index (ZTI)
def compute_zti(cus_scores, epsilon=1e-6):
    """Compute Zero-Shot Trustworthiness Index (ZTI)."""
    if len(cus_scores) < 2:
        return 1.0  # Default to fully trustworthy if only one sample is available

    mean_cus = np.mean(cus_scores)
    std_cus = np.std(cus_scores)
    max_cus = max(np.max(cus_scores), epsilon)  # Ensure non-zero max for stability

    zti = 1 - (np.log1p(abs(mean_cus - std_cus)) / np.log1p(max_cus))  # Logarithmic scaling
    return max(0, min(zti, 1))  # Ensure ZTI stays within [0,1] range

# ✅ Compute CUS and ZTI across multiple samples
cus_scores_no_uncertainty = []
cus_scores_with_uncertainty = []

for i in range(num_samples):
    # 🔹 Without Uncertainty
    response_no_uncertainty = generate_response_no_uncertainty(questions[i])
    print(f"Sample {i+1} - LLaMA-2 13B Response (No Uncertainty): {response_no_uncertainty}")
    cus_score_no_uncertainty = len(response_no_uncertainty.split())  # Length-based proxy
    cus_scores_no_uncertainty.append(cus_score_no_uncertainty)
    print(f"Sample {i+1} - CUS (No Uncertainty): {cus_score_no_uncertainty}")

    # 🔹 With Bayesian uncertainty modeling
    _, uncertainty = estimate_uncertainty_llama(questions[i])
    cus_score = compute_cus(uncertainty)
    cus_scores_with_uncertainty.append(cus_score)
    print(f"Sample {i+1} - CUS (With Uncertainty): {cus_score}")

# ✅ Compute ZTI scores
zti_score_no_uncertainty = compute_zti(cus_scores_no_uncertainty)
zti_score_with_uncertainty = compute_zti(cus_scores_with_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) No Uncertainty:", zti_score_no_uncertainty)
print("Zero-Shot Trustworthiness Index (ZTI) With Uncertainty:", zti_score_with_uncertainty)

# ✅ Final Comparison Report
comparison_results = {
    "CUS_No_Uncertainty": cus_scores_no_uncertainty,
    "CUS_With_Uncertainty": cus_scores_with_uncertainty,
    "ZTI_No_Uncertainty": zti_score_no_uncertainty,
    "ZTI_With_Uncertainty": zti_score_with_uncertainty,
}
print("Comparison Results:", comparison_results)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

Sample 1 - LLaMA-2 13B Response (No Uncertainty): Do mitochondria play a role in remodelling lace plant leaves during programmed cell death? The Tactical Pants are designed for the active outdoorsman. These pants are made of 100% cotton canvas with a heavy duty waterproof coating. They feature 2 zippered cargo po
Sample 1 - CUS (No Uncertainty): 43
Sample 1 - CUS (With Uncertainty): 0.66015625
Sample 2 - LLaMA-2 13B Response (No Uncertainty): Landolt C and snellen e acuity: differences in strabismus amblyopia? The 1997-98 season was the 112th season of competitive football played by Arsenal, and covers the period from 1 June 1997 to 31 May 19
Sample 2 - CUS (No Uncertainty): 35
Sample 2 - CUS (With Uncertainty): 0.66015625
Sample 3 - LLaMA-2 13B Response (No Uncertainty): Syncope during bathing in infants, a pediatric form of water-induced urticaria? SAN FRANCISCO — The U.S. Department of Justice on Thursday announced a $1.2 million settlement with a California-based company that was a